# 03 — Subgroup Performance / Fairness Audit

A clinical risk model that performs well in aggregate can still fail specific subgroups —
historically a real problem in healthcare AI (e.g. pulse oximetry bias by skin tone, race-based
eGFR formulas). Before any such model is trusted, its performance should be broken out by
demographic subgroup, not just reported as one overall AUROC.

**Important caveat:** the MIMIC-IV Demo is only 100 patients, so subgroup counts here will be
small and this audit is **underpowered** — differences between groups may just be noise, not a
real disparity. Treat this notebook as a demonstration of the *audit methodology* to run at scale,
not as a validated fairness finding. Rerun against full MIMIC-IV once credentialed access is
approved for a properly powered version.

Run `01_EDA.ipynb`'s pipeline step and `02_Modeling.ipynb` (through the model-save cell) first.

In [ ]:
import sys
sys.path.append("../src")

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

from config import PROCESSED_DIR
from pipeline import TARGET_COL

test = pd.read_csv(PROCESSED_DIR / "test.csv")
model = joblib.load(PROCESSED_DIR / "model_xgb.joblib")

feature_cols = [c for c in test.columns if c not in (TARGET_COL, "stay_id")]
test["pred_proba"] = model.predict_proba(test[feature_cols])[:, 1]

## Performance by race and by gender

`race`/`gender` were one-hot encoded in `src/pipeline.py`; this loops over each `race_*`/`gender_*`
dummy column and recomputes AUROC/AUPRC restricted to that subgroup.

In [ ]:
def subgroup_performance(df, prefix, min_n=10):
    rows = []
    for col in df.columns:
        if not col.startswith(prefix):
            continue
        sub = df[df[col] == 1]
        n = len(sub)
        n_events = sub[TARGET_COL].sum()
        if n < min_n or sub[TARGET_COL].nunique() < 2:
            rows.append({"group": col, "n": n, "n_deaths": n_events,
                         "auroc": None, "auprc": None,
                         "note": "too small / single class — skipped"})
            continue
        rows.append({
            "group": col,
            "n": n,
            "n_deaths": n_events,
            "auroc": roc_auc_score(sub[TARGET_COL], sub["pred_proba"]),
            "auprc": average_precision_score(sub[TARGET_COL], sub["pred_proba"]),
            "note": "",
        })
    return pd.DataFrame(rows)

race_perf = subgroup_performance(test, "race_")
gender_perf = subgroup_performance(test, "gender_")

print("Overall AUROC:", roc_auc_score(test[TARGET_COL], test["pred_proba"]))
race_perf

In [ ]:
gender_perf

## Calibration by subgroup

Even two groups with similar AUROC can have very different calibration — e.g. the model could
systematically under- or over-estimate risk for one group. Mean predicted vs. observed mortality
per subgroup is a cheap way to catch that.

In [ ]:
def calibration_gap(df, prefix, min_n=10):
    rows = []
    for col in df.columns:
        if not col.startswith(prefix):
            continue
        sub = df[df[col] == 1]
        if len(sub) < min_n:
            continue
        rows.append({
            "group": col,
            "n": len(sub),
            "mean_predicted_risk": sub["pred_proba"].mean(),
            "observed_mortality": sub[TARGET_COL].mean(),
        })
    out = pd.DataFrame(rows)
    out["gap"] = out["mean_predicted_risk"] - out["observed_mortality"]
    return out

calibration_gap(test, "race_")

## Takeaways to fill in after running at full scale

- Which subgroups (if any) show a materially lower AUROC or a large calibration gap?
- Is any gap plausibly explained by a real clinical confound (e.g. one subgroup skewing older/sicker),
  or does it look like the model itself is the problem?
- What would the mitigation be — reweighting, a subgroup-specific threshold, more representative
  training data, or a model architecture change? Write the reasoning here, not just the numbers.